# 9. Полный Transformer — Encoder-Decoder

**Цель:** Собрать полную seq2seq архитектуру трансформера: энкодер, декодер, эмбеддинги, выходную голову. Продемонстрировать авторегрессивную генерацию на задаче копирования.

---

In [2]:
# Teacher forcing: during training, the decoder receives the true target sequence (shifted right), not its own predictions — this makes training faster and more stableimport sys, os, logging, math
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("transformer")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
log.info("Using device: %s", device)

NameError: name 'logging' is not defined

## 9.1 Компоненты трансформера

Переиспользуем все реализации из предыдущих ноутбуков в одном классе.

In [3]:
# Teacher forcing: decoder sees true tokens during training, not its own predictions — enables parallel computation and faster convergence# Causal mask in self-attention prevents the decoder from attending to future positions (autoregressive property)log.debug("Assembling all transformer components")

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):
        batch = Q.size(0)
        Q = self.W_Q(Q).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(F.softmax(scores, dim=-1))
        output = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch, -1, self.d_model)
        return self.W_O(output)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = x + self.dropout1(self.attention(self.norm1(x), self.norm1(x), self.norm1(x), mask))
        x = x + self.dropout2(self.ffn(self.norm2(x)))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Causal self-attention: True = attend, False = masked
        seq_len = x.size(1)
        causal = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).bool()  # lower-tri = attend
        if tgt_mask is not None:
            # И позиция должна быть валидна (не pad) И не в будущем
            combined_mask = tgt_mask.unsqueeze(1).unsqueeze(2) & causal.unsqueeze(0).unsqueeze(0)
        else:
            combined_mask = causal.unsqueeze(0).unsqueeze(0)
        x = x + self.dropout1(self.self_attention(self.norm1(x), self.norm1(x), self.norm1(x), combined_mask))
        x = x + self.dropout2(self.cross_attention(self.norm2(x), self.norm2(encoder_output), self.norm2(encoder_output), src_mask))
        x = x + self.dropout3(self.ffn(self.norm3(x)))
        return x

log.debug("All components defined")

NameError: name 'nn' is not defined

## 9.2 PositionalEncoding

In [ ]:
# Positional encoding gives the transformer a sense of token order — the model has no inherent notion of position without itclass PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

## 9.3 Полный Transformer

Объединяем всё: эмбеддинги, PE, энкодер, декодер, output head.

In [ ]:
# Difference between training and inference:# - Training (forward): teacher forcing — decoder receives the true target sequence shifted right, entire output computed in one pass# - Inference (generate): autoregressive — decoder feeds its own previous prediction back as the next input, one token at a timelog.debug("Implementing full Transformer model")

class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_encoder_layers,
                 num_decoder_layers, d_ff=None, max_len=5000, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Embeddings + Positional Encoding
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Encoder
        self.encoder_layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])
        
        # Decoder
        self.decoder_layers = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])
        
        # Output head
        self.output_proj = nn.Linear(d_model, vocab_size)
        
        # Инициализация
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        
        log.info("Transformer: vocab=%d, d_model=%d, enc=%d, dec=%d, params=%d",
                 vocab_size, d_model, num_encoder_layers, num_decoder_layers,
                 sum(p.numel() for p in self.parameters()))
    
    def encode(self, src, src_mask=None):
        x = self.pos_encoding(self.embedding(src) * math.sqrt(self.d_model))
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return x
    
    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        x = self.pos_encoding(self.embedding(tgt) * math.sqrt(self.d_model))
        for layer in self.decoder_layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return x
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encode(src, src_mask)
        dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask)
        return self.output_proj(dec_output)

log.debug("Transformer class defined")

In [ ]:
# Forward pass with teacher forcing: decoder gets true target tokens (shifted right), not model predictions — all positions computed in parallellog.debug("Creating small transformer for testing")

vocab_size = 20
model = Transformer(
    vocab_size=vocab_size,
    d_model=32,
    n_heads=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=64,
    max_len=50,
).to(device)

src = torch.randint(0, vocab_size, (4, 10)).to(device)  # (batch, src_len)
tgt = torch.randint(0, vocab_size, (4, 8)).to(device)   # (batch, tgt_len)

output = model(src, tgt)
print(f"Source shape:          {src.shape}")
print(f"Target shape:          {tgt.shape}")
print(f"Output shape:          {output.shape}")
print(f"Output dim:            {output.shape[-1]} (should be {vocab_size})")
log.info("Full transformer forward pass OK")

### Teacher forcing: обучение с учителем vs авторегрессия на инференсе

**Teacher forcing (обучение):**
```
Вход декодера: [BOS, t1, t2, t3]  (истинная последовательность)
Предсказание:  [p1, p2, p3, EOS]
Ошибка:        cross_entropy(p1, t1) + cross_entropy(p2, t2) + ...
```
- Декодер на каждом шаге получает **правильный предыдущий токен**, а не свой собственный
- Все позиции вычисляются **параллельно** — один forward вместо цикла
- Обучение быстрое и стабильное

**Авторегрессия (инференс):**
```
Шаг 1: [BOS]              → предсказание p1
Шаг 2: [BOS, p1]           → предсказание p2
Шаг 3: [BOS, p1, p2]       → предсказание p3
...
```
- Каждый следующий токен зависит от **собственных предыдущих предсказаний**
- Ошибки на ранних шагах накапливаются (exposure bias)
- Генерация последовательная — нельзя распараллелить

> Teacher forcing на трэйне, авторегрессия на инференсе — стандартная практика для seq2seq моделей.

## 9.4 Демонстрация: задача копирования

Обучим трансформер копировать последовательности.

In [ ]:
# The copy task: why copying simple sequences tests whether the model learned the attention mechanism# The model must align each output position to the corresponding input position via cross-attention# If attention is broken or misaligned, the model cannot copy — making this an ideal minimal diagnostic tasklog.debug("Preparing copy task data")

from torch.utils.data import DataLoader, TensorDataset

BOS, EOS, PAD = 0, 1, 2  # специальные токены

def generate_copy_data(num_samples, max_len, vocab_size):
    """Генерирует пары src-tgt для задачи копирования."""
    src_list, tgt_list = [], []
    for _ in range(num_samples):
        length = np.random.randint(2, max_len + 1)
        seq = np.random.randint(3, vocab_size, size=length).tolist()
        src = [BOS] + seq + [EOS]
        src = src + [PAD] * (max_len + 2 - len(src))
        
        tgt_in = [BOS] + seq + [EOS]
        tgt_in = tgt_in + [PAD] * (max_len + 2 - len(tgt_in))
        
        tgt_out = seq + [EOS]
        tgt_out = tgt_out + [PAD] * (max_len + 2 - len(tgt_out))
        
        src_list.append(src)
        tgt_list.append(tgt_out)
    
    return (torch.tensor(src_list), torch.tensor(tgt_list))

vocab_size = 16
max_len = 6
train_src, train_tgt = generate_copy_data(500, max_len, vocab_size)
print(f"Train src shape: {train_src.shape}")
print(f"Train tgt shape: {train_tgt.shape}")
print(f"Source example:   {train_src[0].tolist()}")
print(f"Target example:   {train_tgt[0].tolist()}")
log.info("Copy task data generated")

### Копирование последовательностей: почему это тестовая задача

Задача копирования — простейший тест seq2seq модели:

- **Вход:** последовательность токенов `[A, B, C]`
- **Выход:** та же последовательность `[A, B, C]`

Почему она полезна:
1. **Минимальная проверка внимания (attention):** модель должна выучить alignment между i-й позицией входа и i-й позицией выхода. Если cross-attention не работает — модель не сможет скопировать.
2. **Нет внешних знаний:** не нужны словари, грамматика, семантика — только способность "посмотреть" на вход.
3. **Известный oracle:** мы точно знаем, каким должен быть выход — метрика качества (accuracy) прозрачна.
4. **Диагностика:** если модель не сходится на копировании, она тем более не сработает на реальных данных.

Ожидаемый результат: после обучения loss ≈ 0, генерация ≈ 100% совпадение.

In [ ]:
# Teacher forcing: decoder input = BOS + target[:-1] (true prefix, not predicted tokens)# During training the model sees correct history at every timestep — dramatically faster and more stable than autoregressive training# The copy task requires the model to learn proper encoder-decoder alignment through cross-attentionlog.debug("Training transformer on copy task")

model_small = Transformer(
    vocab_size=vocab_size,
    d_model=32,
    n_heads=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=64,
    max_len=max_len + 2,
    dropout=0.1,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.Adam(model_small.parameters(), lr=0.001)

n_epochs = 50
batch_size = 32
losses = []

for epoch in range(n_epochs):
    epoch_loss = 0
    n_batches = 0
    perm = torch.randperm(len(train_src))
    
    for i in range(0, len(train_src), batch_size):
        idx = perm[i:i+batch_size]
        src = train_src[idx].to(device)
        tgt = train_tgt[idx].to(device)
        
        # Teacher forcing: decoder input = BOS + target[:-1]
        dec_input = torch.cat([
            torch.full((len(idx), 1), BOS, device=device, dtype=torch.long),
            tgt[:, :-1]
        ], dim=1)
        
        output = model_small(src, dec_input)
        loss = criterion(output.reshape(-1, vocab_size), tgt.reshape(-1))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_small.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    
    if epoch % 10 == 0:
        log.info("Epoch %d: loss=%.4f", epoch, avg_loss)

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss: Copy Task')
plt.grid(True)
plt.show()
log.info("Training complete, final loss=%.4f", losses[-1])

## 9.5 Авторегрессивная генерация

Генерируем последовательность по одному токену.

In [ ]:
# Inference mode: no teacher forcing — the model feeds its own previous prediction back as the next input (autoregressive)# Iterative decoding: one token at a time, each step conditions on all previously generated tokens# Greedy decoding picks the most likely token at each step (no beam search)# Key difference from training: errors compound during inference — the model sees its own mistakes, not ground truthlog.debug("Implementing autoregressive generation")

@torch.no_grad()
def generate(model, src, max_len=10):
    model.eval()
    src = src.to(device)
    enc_output = model.encode(src)
    
    # Начинаем с BOS
    tgt = torch.full((src.size(0), 1), BOS, dtype=torch.long, device=device)
    
    for i in range(max_len):
        dec_output = model.decode(tgt, enc_output)
        logits = model.output_proj(dec_output[:, -1:, :])  # только последний токен
        next_token = logits.argmax(dim=-1)  # greedy decoding
        tgt = torch.cat([tgt, next_token], dim=1)
        
        # Стоп при EOS
        if (next_token == EOS).all():
            break
    
    return tgt

test_src, _ = generate_copy_data(4, 4, vocab_size)
for i, src in enumerate(test_src[:3]):
    src = src.unsqueeze(0).to(device)
    pred = generate(model_small, src, max_len=12)
    src_tokens = [t for t in src[0].tolist() if t not in (BOS, EOS, PAD)]
    pred_tokens = [t for t in pred[0].tolist() if t not in (BOS, EOS, PAD)]
    print(f"Sample {i+1}: src={src_tokens}, pred={pred_tokens}, match={src_tokens == pred_tokens}")
log.info("Generation demo complete")

In [ ]:
# Teacher forcing enables fast parallel training; autoregressive decoding handles generation at inference# The copy task validates that the full encoder-decoder attention mechanism works correctlyprint("=== Full Transformer complete ===")
print("Topics covered:")
print("  - Full Transformer: encoder + decoder + output head")
print("  - Tokenization with special tokens (BOS, EOS, PAD)")
print("  - Embeddings + PositionalEncoding")
print("  - Copy task: training with teacher forcing")
print("  - Autoregressive generation (greedy decoding)")
print(f"  - Final loss: {losses[-1]:.4f}")
log.info("Full transformer notebook complete")

📚 **Полезные ссылки:**
- [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762)
- [The Annotated Transformer (Harvard NLP)](http://nlp.seas.harvard.edu/2018/04/03/attention.html)
